# open streat map buildings

#### how to download:
downloading format is osm.pbf, from https://download.geofabrik.de/

convert it to geoJSON:
download osmium:

    sudo apt-get install osmium-tool

In [ ]:
run this line:

    osmium export path/to/file.osm.pbf -f geojson -o output.geojson

now run this cell to filter the buildings:

In [5]:
import geojson

# Load GeoJSON and initialize lists
buildings = []
nodes = set()  # Use a set to avoid duplicates

with open('output.geojson') as f:
    data = geojson.load(f)
    for feat:ure in data['features']:
        # Check if the feature represents a building
        if 'building' in feature['properties']:
            buildings.append(feature)  # Add the building

            # Collect all nodes related to this feature
            if 'coordinates' in feature['geometry']:
                coordinates = feature['geometry']['coordinates']
                
                # Handle different geometry types
                if feature['geometry']['type'] == 'Polygon':
                    for coord in coordinates:
                        nodes.update(tuple(coord_point) for coord_point in coord)
                elif feature['geometry']['type'] == 'MultiPolygon':
                    for polygon in coordinates:
                        for coord in polygon:
                            nodes.update(tuple(coord_point) for coord_point in coord)
                # Add more geometry types as needed

# Save filtered buildings with their nodes
output_geojson = {
    'type': 'FeatureCollection',
    'features': buildings,
    'nodes': list(nodes)  # Convert set back to list
}

with open('filtered_buildings.geojson', 'w') as f:
    geojson.dump(output_geojson, f)


filtered_buildings.geojson format:

In [ ]:
"type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {
        "building": "yes",
        "name": "Building A"
      },
      "geometry": {
        "type": "Polygon",
        "coordinates": [
          [
            [34.7980, 32.1138],
            [34.7990, 32.1140],
            [34.7990, 32.1130],
            [34.7980, 32.1138]
          ]
        ]
      }
    }
  ],
  "nodes": [
    [34.7980, 32.1138],
    [34.7990, 32.1140],
    [34.7990, 32.1130]
  ]
}

now, add it to the project:

In [3]:
from hera import Project 
from hera import toolkitHome
from hera import toolkit

from hera import *
import geopandas as gpd


proj = Project()

from hera.utils.logging import initialize_logging,with_logger
initialize_logging(
          with_logger("hera.measurements.GIS.vector.buildings.analysis", handlers=['console'], level='INFO', propagate=False)
    )

tk = toolkitHome.getToolkit(toolkitName=toolkitHome.GIS_BUILDINGS)

tk.addDataSource(dataSourceName="OSM",
                 resource="/home/shira/filtered_buildings.geojson",
                 dataFormat=tk.datatypes.JSON_DICT,
                 version=(0,0,1),overwrite=True)

doc = tk.getDataSourceDocument(datasourceName="OSM")
data = doc.getData()

## filter area
The function filter_buildings_in_area takes a GeoJSON dictionary, filters the buildings in the desired area, and converts the result to a GeoPandas DataFrame.

In [8]:
min_lon, min_lat, max_lon, max_lat =34.75937, 32.07038, 34.77344, 32.09400  # Example coordinates

In [21]:
filtered_bulidings = tk.filter_buildings_in_area(data,min_lon, min_lat, max_lon, max_lat)
filtered_bulidings

,addr:city:en,addr:housenumber,addr:street,amenity,brand,brand:en,brand:he,brand:wikidata,building,cuisine,...,name:ar,air_conditioning,contact:facebook,contact:instagram,contact:linkedin,max_level,min_level,start_date,check_date,source:height
0,Tel Aviv,172,בן יהודה,cafe,קפה קפה,Cafe Cafe,קפה קפה,Q5017233,yes,coffee_shop,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Tel Aviv,15,ארלוזורוב,pub,NaN,NaN,NaN,NaN,yes,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,yes,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,place_of_worship,NaN,NaN,NaN,NaN,yes,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,yes,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3341,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,yes,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3342,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,yes,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3343,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,yes,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3344,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,yes,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## get heights
The function get_buildings_height takes a GeoPandas DataFrame as input and returns the heights of the buildings, along with their names and coordinates.

In [22]:
buildings_height = tk.get_buildings_height(filtered_bulidings)
buildings_height.dropna()

,name,geometry,height
7,מלון קרלטון,"LINESTRING (34.76943 32.08673, 34.76977 32.086...",56
9,מלון קרלטון,"MULTIPOLYGON (((34.76933 32.08643, 34.76967 32...",56
150,מלון הרודס תל אביב,"LINESTRING (34.76902 32.08469, 34.76896 32.084...",69
152,מלון קראון פלאזה,"LINESTRING (34.76873 32.08401, 34.76866 32.083...",70
160,מגדל האופרה,"LINESTRING (34.76542 32.07397, 34.76546 32.073...",4.7
176,מגדל ישרוטל,"LINESTRING (34.76732 32.07679, 34.76731 32.076...",70.8
178,מלון מטרופוליטן,"LINESTRING (34.76732 32.07566, 34.76743 32.075...",62
319,מלון הרודס תל אביב,"MULTIPOLYGON (((34.76896 32.08452, 34.76900 32...",69
321,מלון קראון פלאזה,"MULTIPOLYGON (((34.76866 32.08383, 34.76870 32...",70
329,מגדל האופרה,"MULTIPOLYGON (((34.76542 32.07397, 34.76546 32...",4.7
